# Feature generation | Google Earth Engine

Public adaptation of `src/06_two_dates/03_prepare_figure2_contrasting_dates_colab.ipynb` in the team archive. Use this to inspect how features were produced or to regenerate them with your own Earth Engine project and authorized inputs. To use the existing exports, go directly to the prediction notebooks instead. Exports are disabled by default. Details: [Feature sources and Drive inputs](https://github.com/Boonyaratt/sentinel2-water-quality/blob/main/docs/drive-features-th.md).


# Figure 2 Data Preparation: Contrasting Contextual Conditions

This single Google Colab notebook combines date-pair screening and two-date export for **Thale Luang (WB02)**. It:

1. filters dates that passed the corrected `static_intersection_v2` clear-water screening;
2. extracts the dynamic contextual predictors used by the locked models using only complete hours before Sentinel-2 acquisition;
3. ranks date pairs without using model predictions;
4. locks one pair of dates;
5. creates the same 62-band moving-window feature stack and RGB export for both dates on one reviewed polygon and one reference grid; and
6. writes the hand-off manifest and readiness audits required by the Figure 2 prediction notebook.

Model fitting and Figure 2 prediction are intentionally not performed here.

In [ ]:
# Colab setup.
from google.colab import drive
drive.mount("/content/drive")

import importlib.util
import subprocess
import sys

required = {
    "ee": "earthengine-api",
    "geemap": "geemap",
    "rasterio": "rasterio",
}
missing = [
    package for module, package in required.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )

In [ ]:
from itertools import combinations
from pathlib import Path
import fnmatch
import json
import re
import time

import ee
import geemap
import numpy as np
import pandas as pd
import rasterio
from IPython.display import display
from google.colab import files

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

## 1. Configuration and inputs

In [ ]:
MYDRIVE = Path("/content/drive/MyDrive")
PROJECT_DIR = MYDRIVE / "wqr_feature_rebuild"

TARGET_MAP_ID = "WB02"
TARGET_WATERBODY = "ทะเลหลวง"

# None uses the highest-ranked pair. To lock a reviewed pair, set:
# DATE_PAIR_OVERRIDE = ["2024-02-05", "2024-07-19"]
DATE_PAIR_OVERRIDE = None
MIN_DATE_SEPARATION_DAYS = 30

# Review the ranking before changing this to True.
START_EXPORT_TASKS = False
WAIT_FOR_EXPORTS = False
MAX_WAIT_MINUTES = 180
POLL_SECONDS = 30

COVERAGE_PATH_OVERRIDE = None
CONTEXT_PATH_OVERRIDE = None
POLYGON_PATH_OVERRIDE = None
REFERENCE_GRID_TIF_OVERRIDE = None

COVERAGE_PATTERN = "clear_water_coverage_by_date_waterbody*.csv"
CONTEXT_FILENAME = "rswq_context_features_ctx_2562_2568_gsmap_era5h_strict_v2.csv"
POLYGON_FILENAME = "waterbody_polygons.geojson"
REFERENCE_GRID_FILENAME = "WB02_20240719_moving_window_features.tif"

EE_PROJECT = "YOUR_EARTH_ENGINE_PROJECT_ID"
if EE_PROJECT == "YOUR_EARTH_ENGINE_PROJECT_ID":
    raise ValueError("Set EE_PROJECT to a project you can use in Earth Engine.")
S2_COLLECTION = "COPERNICUS/S2_SR_HARMONIZED"
GSW_IMAGE = "JRC/GSW1_4/GlobalSurfaceWater"
GSMAP_COLLECTION = "JAXA/GPM_L3/GSMaP/v8/operational"
ERA5_COLLECTION = "ECMWF/ERA5_LAND/HOURLY"
WORLDCOVER_COLLECTION = "ESA/WorldCover/v200"

MOVING_WINDOW_RADIUS_M = 100
EXPORT_SCALE_M = 20
MIN_WINDOW_WATER_PIXELS = 3
MAX_SCENE_CLOUD_PCT = 80
JRC_OCCURRENCE_THRESHOLD = 70

DRIVE_EXPORT_FOLDER = "Water_Quality_RS_Figure2_Features_rebuild"
RGB_DRIVE_EXPORT_FOLDER = "Water_Quality_RS_Figure2_RGB_rebuild"
OUTPUT_DIR = PROJECT_DIR / "Maps" / "figure2_context_contrast" / "preparation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def resolve_file(override, exact_name=None, pattern=None, required=True):
    if override:
        path = Path(override)
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    roots = [
        Path("/content"), PROJECT_DIR, MYDRIVE,
        MYDRIVE / "Water Quality Research",
    ]
    matches = []
    for root in roots:
        if not root.exists():
            continue
        if exact_name:
            matches.extend(root.rglob(exact_name))
        elif pattern:
            matches.extend(root.rglob(pattern))
    matches = sorted(
        set(matches), key=lambda path: path.stat().st_mtime, reverse=True
    )
    if not matches:
        if not required:
            return None
        label = exact_name or pattern
        print(f"Could not find {label}. Upload it now.")
        uploaded = files.upload()
        uploaded_paths = [Path("/content") / name for name in uploaded]
        matches = [
            path for path in uploaded_paths
            if (
                path.name == exact_name if exact_name
                else fnmatch.fnmatch(path.name, pattern)
            )
        ]
    if not matches:
        raise FileNotFoundError(exact_name or pattern)
    if len(matches) > 1:
        print("Multiple matches found; using newest:", matches[0])
    return matches[0]


COVERAGE_PATH = resolve_file(
    COVERAGE_PATH_OVERRIDE, pattern=COVERAGE_PATTERN
)
CONTEXT_PATH = resolve_file(
    CONTEXT_PATH_OVERRIDE, exact_name=CONTEXT_FILENAME
)
POLYGON_PATH = resolve_file(
    POLYGON_PATH_OVERRIDE, exact_name=POLYGON_FILENAME
)
REFERENCE_GRID_PATH = resolve_file(
    REFERENCE_GRID_TIF_OVERRIDE,
    exact_name=REFERENCE_GRID_FILENAME,
    required=True,
)

print("Coverage:", COVERAGE_PATH)
print("Context:", CONTEXT_PATH)
print("Polygon:", POLYGON_PATH)
print("Reference grid:", REFERENCE_GRID_PATH)
print("Output:", OUTPUT_DIR)

## 2. Earth Engine authentication

In [ ]:
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine initialized:", EE_PROJECT)

## 3. Candidate dates, stations, and reviewed polygon

In [ ]:
DYNAMIC_CONTEXT_FEATURES = [
    "rain_gsmap_prior_24h_mm",
    "rain_gsmap_prior_336h_mm",
    "rain_gsmap_prior_720h_mm",
    "rainy_hours_gsmap_prior_336h",
    "era5h_runoff_prior_1h_mm",
    "era5h_runoff_prior_24h_mm",
    "era5h_air_temp_prior_72h_C",
    "era5h_wind_max_prior_168h_m_s",
    "era5h_solar_prior_168h_MJ_m2",
]

CONTEXT_GROUPS = {
    "rainfall": [
        "rain_gsmap_prior_24h_mm",
        "rain_gsmap_prior_336h_mm",
        "rain_gsmap_prior_720h_mm",
        "rainy_hours_gsmap_prior_336h",
    ],
    "runoff": [
        "era5h_runoff_prior_1h_mm",
        "era5h_runoff_prior_24h_mm",
    ],
    "meteorology": [
        "era5h_air_temp_prior_72h_C",
        "era5h_wind_max_prior_168h_m_s",
        "era5h_solar_prior_168h_MJ_m2",
    ],
}

coverage = pd.read_csv(COVERAGE_PATH, low_memory=False)
context_columns = [
    "sample_id", "station_canonical", "coordinate_waterbody",
    "latitude", "longitude", "field_date",
    *DYNAMIC_CONTEXT_FEATURES,
]
context = pd.read_csv(
    CONTEXT_PATH, usecols=context_columns, low_memory=False
)

required_coverage = {
    "sat_date", "coordinate_waterbody", "screening_metric_version",
    "clear_static_pixels", "clear_static_water_pixels",
    "clear_static_coverage_pct", "clear_static_water_coverage_pct",
    "eligible_static_water", "clear_water_ready",
}
missing = required_coverage.difference(coverage.columns)
if missing:
    raise KeyError(f"Coverage table is missing: {sorted(missing)}")
versions = set(coverage["screening_metric_version"].dropna().astype(str))
if versions != {"static_intersection_v2"}:
    raise ValueError(f"Expected static_intersection_v2, found {versions}")

coverage["sat_date"] = pd.to_datetime(
    coverage["sat_date"], errors="coerce"
).dt.normalize()
for column in ["latitude", "longitude", *DYNAMIC_CONTEXT_FEATURES]:
    context[column] = pd.to_numeric(context[column], errors="coerce")

candidates = coverage.loc[
    coverage["coordinate_waterbody"].eq(TARGET_WATERBODY)
    & coverage["eligible_static_water"].eq(True)
    & coverage["clear_water_ready"].eq(True)
    & coverage["sat_date"].notna()
].copy()
candidates = candidates.sort_values("sat_date").reset_index(drop=True)
if len(candidates) < 2:
    raise ValueError("At least two clear-water-ready dates are required.")

station_inventory = (
    context.loc[
        context["coordinate_waterbody"].eq(TARGET_WATERBODY)
        & context["station_canonical"].notna()
        & context["latitude"].notna()
        & context["longitude"].notna()
    ]
    .drop_duplicates(
        ["station_canonical", "latitude", "longitude"]
    )
)

polygon_geojson = json.loads(
    Path(POLYGON_PATH).read_text(encoding="utf-8")
)
polygon_matches = [
    feature for feature in polygon_geojson.get("features", [])
    if str(feature.get("properties", {}).get("map_id")) == TARGET_MAP_ID
]
if len(polygon_matches) != 1:
    raise ValueError(
        f"Expected one reviewed polygon for {TARGET_MAP_ID}; "
        f"found {len(polygon_matches)}"
    )
polygon_feature = polygon_matches[0]
if polygon_feature.get("geometry", {}).get("type") not in {
    "Polygon", "MultiPolygon"
}:
    raise ValueError("Reviewed AOI must be Polygon or MultiPolygon.")
TARGET_GEOMETRY = ee.Geometry(polygon_feature["geometry"])

display(candidates[[
    "sat_date", "clear_static_water_coverage_pct",
    "clear_static_coverage_pct", "clear_static_water_pixels",
]].round(3))
print("Candidate dates:", len(candidates))
print("Station points:", station_inventory["station_canonical"].nunique())
print("Polygon area (km2):", TARGET_GEOMETRY.area(maxError=1).divide(1e6).getInfo())

## 4. Corrected Sentinel-2 preprocessing

In [ ]:
S2_BANDS = ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B9", "B11", "B12"]
BASE_FEATURES = S2_BANDS + ["NDWI", "MNDWI", "NDTI", "NDCI", "NDVI", "AWEIsh"]


def safe_ratio(numerator, denominator, name):
    return numerator.divide(denominator).updateMask(denominator.neq(0)).rename(name)


def normalized_difference(a, b, name):
    return a.subtract(b).divide(a.add(b)).updateMask(a.add(b).neq(0)).rename(name)


def prepare_s2(image):
    scl = image.select("SCL")
    clear = (
        scl.neq(0).And(scl.neq(1)).And(scl.neq(3)).And(scl.neq(8))
        .And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    )
    sr = image.select(S2_BANDS).multiply(0.0001).rename(S2_BANDS)
    blue = sr.select("B2")
    green = sr.select("B3")
    red = sr.select("B4")
    red_edge1 = sr.select("B5")
    nir = sr.select("B8")
    swir1 = sr.select("B11")
    swir2 = sr.select("B12")

    ndvi = normalized_difference(nir, red, "NDVI")
    ndwi = normalized_difference(green, nir, "NDWI")
    mndwi = normalized_difference(green, swir1, "MNDWI")
    ndti = normalized_difference(red, green, "NDTI")
    ndci = normalized_difference(red_edge1, red, "NDCI")
    ndbi = normalized_difference(swir1, nir, "NDBI")
    aweish = (
        blue.add(green.multiply(2.5))
        .subtract(nir.add(swir1).multiply(1.5))
        .subtract(swir2.multiply(0.25))
        .rename("AWEIsh")
    )
    brightness = blue.add(green).add(red).divide(3)

    built_or_bridge = (
        brightness.gt(0.12).And(mndwi.lt(0.12)).And(ndwi.lt(0.12)).And(aweish.lt(0.08))
    ).Or(
        ndbi.gt(0).And(mndwi.lt(0.15))
    ).Or(
        nir.gt(0.20).And(swir1.gt(0.18)).And(mndwi.lt(-0.10)).And(scl.neq(6))
    )

    jrc_water = ee.Image(GSW_IMAGE).select("occurrence").gte(JRC_OCCURRENCE_THRESHOLD)
    jrc_current = (
        jrc_water.And(scl.eq(6).Or(ndwi.gt(0.03)).Or(mndwi.gt(0.03)).Or(aweish.gt(0.03)))
        .And(nir.lt(0.25)).And(swir1.lt(0.18)).And(ndvi.lt(0.30)).And(built_or_bridge.Not())
    )
    strong = mndwi.gt(0.20).And(aweish.gt(0.05)).And(nir.lt(0.12)).And(swir1.lt(0.10)).And(ndvi.lt(0.15))
    turbid = mndwi.gt(-0.20).And(ndwi.gt(-0.30)).And(aweish.gt(-0.10)).And(nir.lt(0.20)).And(swir1.lt(0.16)).And(ndvi.lt(0.30))
    scl_current = (
        scl.eq(6).And(nir.lt(0.16)).And(swir1.lt(0.12)).And(ndvi.lt(0.22))
        .And(mndwi.gt(0).Or(ndwi.gt(-0.05)).Or(aweish.gt(-0.02))).And(built_or_bridge.Not())
    )
    index_core = strong.Or(turbid.And(mndwi.gt(0.05).Or(aweish.gt(0.02)))).And(built_or_bridge.Not())
    water_core = scl_current.Or(jrc_current).Or(index_core).And(built_or_bridge.Not())
    water_candidate = turbid.And(built_or_bridge.Not())
    core_neighborhood = water_core.focal_max(radius=2, units="pixels")
    water_mask = water_core.Or(water_candidate.And(core_neighborhood)).And(built_or_bridge.Not()).updateMask(clear)

    return (
        sr.addBands([ndwi, mndwi, ndti, ndci, ndvi, aweish])
        .addBands(ee.Image(1).rename("valid_mask").updateMask(clear))
        .addBands(water_mask.rename("water_mask"))
        .copyProperties(image, ["system:index", "system:time_start", "CLOUDY_PIXEL_PERCENTAGE"])
    )

## 5. Moving-window and environmental feature construction

In [ ]:
# Every neighborhood uses an explicit pixel kernel on an explicit projection.
# This prevents GEE from converting a meter kernel on an unintended native scale.
MOVING_WINDOW_RADIUS_PIXELS = int(np.ceil(MOVING_WINDOW_RADIUS_M / EXPORT_SCALE_M))
WINDOW_KERNEL = ee.Kernel.circle(
    radius=MOVING_WINDOW_RADIUS_PIXELS, units="pixels", normalize=False
)
JRC_CONTEXT_SCALE_M = 30
JRC_500_RADIUS_PIXELS = int(np.ceil(500 / JRC_CONTEXT_SCALE_M))
JRC_500_KERNEL = ee.Kernel.circle(
    radius=JRC_500_RADIUS_PIXELS, units="pixels", normalize=False
)
COARSE_CONTEXT_SCALE_M = 11_132
WORLDCOVER = ee.ImageCollection(WORLDCOVER_COLLECTION).first().select("Map")
# Aggregate 10 m WorldCover indicators to 50 m before large focal windows.
# A direct 5 km radius at 10 m creates a 1001-pixel kernel; GEE allows at most 512.
WORLDCOVER_FRACTION_SCALE_M = 50
JRC = ee.Image(GSW_IMAGE)
GSMAP = ee.ImageCollection(GSMAP_COLLECTION).select("hourlyPrecipRateGC")
ERA5 = ee.ImageCollection(ERA5_COLLECTION)


def focal_mean_coarse(image, radius_m, output_name):
    radius_pixels = int(np.ceil(radius_m / COARSE_CONTEXT_SCALE_M))
    if 2 * radius_pixels + 1 > 512:
        raise ValueError(f"Coarse kernel too large for {output_name}: {2 * radius_pixels + 1} pixels")
    kernel = ee.Kernel.circle(radius=radius_pixels, units="pixels", normalize=False)
    projected = image.reproject(crs="EPSG:4326", scale=COARSE_CONTEXT_SCALE_M)
    return projected.reduceNeighborhood(
        ee.Reducer.mean(), kernel, skipMasked=False
    ).rename(output_name)


def make_static_context():
    jrc_projection = JRC.select("occurrence").projection()
    jrc_occurrence_30m = JRC.select("occurrence").reproject(
        crs=jrc_projection, scale=JRC_CONTEXT_SCALE_M
    )
    jrc_recurrence_30m = JRC.select("recurrence").reproject(
        crs=jrc_projection, scale=JRC_CONTEXT_SCALE_M
    )
    bands = [
        jrc_occurrence_30m.reduceNeighborhood(
            ee.Reducer.mean(), JRC_500_KERNEL, skipMasked=False
        ).rename("jrc_occurrence_mean_500m"),
        jrc_recurrence_30m.reduceNeighborhood(
            ee.Reducer.mean(), JRC_500_KERNEL, skipMasked=False
        ).rename("jrc_recurrence_mean_500m"),
    ]
    selected_wc = {
        ("tree", 10, 500), ("tree", 10, 1000),
        ("shrub", 20, 500), ("shrub", 20, 1000), ("shrub", 20, 5000),
        ("grass", 30, 1000), ("crop", 40, 1000),
        ("bare", 60, 500), ("bare", 60, 1000),
        ("water", 80, 500), ("water", 80, 1000), ("water", 80, 5000),
        ("wetland", 90, 5000),
    }
    for name, class_value, radius in sorted(selected_wc):
        # Mean aggregation preserves class fraction while keeping the 5 km kernel below GEE's limit.
        class_fraction_50m = (
            WORLDCOVER.eq(class_value).unmask(0).toFloat()
            .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=1024)
            .reproject(crs=WORLDCOVER.projection(), scale=WORLDCOVER_FRACTION_SCALE_M)
        )
        # Use an explicit pixel kernel so GEE cannot reinterpret 5 km on the native 10 m grid.
        radius_pixels = int(np.ceil(radius / WORLDCOVER_FRACTION_SCALE_M))
        fraction_kernel = ee.Kernel.circle(
            radius=radius_pixels, units="pixels", normalize=False
        )
        bands.append(
            class_fraction_50m.reduceNeighborhood(
                reducer=ee.Reducer.mean(),
                kernel=fraction_kernel,
                skipMasked=False,
            ).rename(f"wc_{name}_frac_{radius}m")
        )
    return ee.Image.cat(bands)



MAX_KERNEL_DIAMETERS = {
    "moving_window": 2 * MOVING_WINDOW_RADIUS_PIXELS + 1,
    "jrc_500m": 2 * JRC_500_RADIUS_PIXELS + 1,
    "worldcover_5000m": 2 * int(np.ceil(5000 / WORLDCOVER_FRACTION_SCALE_M)) + 1,
    "coarse_15000m": 2 * int(np.ceil(15000 / COARSE_CONTEXT_SCALE_M)) + 1,
}
if max(MAX_KERNEL_DIAMETERS.values()) > 512:
    raise ValueError(f"Kernel guard failed: {MAX_KERNEL_DIAMETERS}")
print("Explicit kernel diameters (pixels):", MAX_KERNEL_DIAMETERS)

STATIC_CONTEXT = make_static_context()


def prep_era5(image):
    air = image.select("temperature_2m").subtract(273.15).rename("air_temp_C")
    u = image.select("u_component_of_wind_10m")
    v = image.select("v_component_of_wind_10m")
    wind = u.pow(2).add(v.pow(2)).sqrt().rename("wind_speed_m_s")
    runoff = image.select("surface_runoff_hourly").multiply(1000).rename("runoff_mm")
    solar = image.select("surface_solar_radiation_downwards_hourly").divide(1e6).rename("solar_MJ_m2")
    return ee.Image.cat([air, wind, runoff, solar]).copyProperties(image, ["system:time_start"])


def make_temporal_context(cutoff):
    # Only complete hours before the Sentinel-2 acquisition-hour cutoff are used.
    rain24 = GSMAP.filterDate(cutoff.advance(-24, "hour"), cutoff).sum()
    rain336 = GSMAP.filterDate(cutoff.advance(-336, "hour"), cutoff)
    rain720 = GSMAP.filterDate(cutoff.advance(-720, "hour"), cutoff).sum()
    rainy336 = rain336.map(lambda img: img.gt(1.0).rename("rainy")).sum()

    era1 = ERA5.filterDate(cutoff.advance(-1, "hour"), cutoff).map(prep_era5)
    era24 = ERA5.filterDate(cutoff.advance(-24, "hour"), cutoff).map(prep_era5)
    era72 = ERA5.filterDate(cutoff.advance(-72, "hour"), cutoff).map(prep_era5)
    era168 = ERA5.filterDate(cutoff.advance(-168, "hour"), cutoff).map(prep_era5)

    coarse = ee.Image.cat([
        focal_mean_coarse(rain24, 10_000, "rain_gsmap_prior_24h_mm"),
        focal_mean_coarse(rain336.sum(), 10_000, "rain_gsmap_prior_336h_mm"),
        focal_mean_coarse(rain720, 10_000, "rain_gsmap_prior_720h_mm"),
        focal_mean_coarse(rainy336, 10_000, "rainy_hours_gsmap_prior_336h"),
        focal_mean_coarse(era1.sum().select("runoff_mm"), 15_000, "era5h_runoff_prior_1h_mm"),
        focal_mean_coarse(era24.sum().select("runoff_mm"), 15_000, "era5h_runoff_prior_24h_mm"),
        focal_mean_coarse(era72.mean().select("air_temp_C"), 15_000, "era5h_air_temp_prior_72h_C"),
        focal_mean_coarse(era168.max().select("wind_speed_m_s"), 15_000, "era5h_wind_max_prior_168h_m_s"),
        focal_mean_coarse(era168.sum().select("solar_MJ_m2"), 15_000, "era5h_solar_prior_168h_MJ_m2"),
    ])
    month = ee.Image.constant(ee.Number.parse(cutoff.format("M"))).rename("month").toFloat()
    doy = ee.Image.constant(cutoff.getRelative("day", "year").add(1)).rename("day_of_year").toFloat()
    return coarse.addBands([month, doy])


def make_feature_stack(row, geometry):
    sat_date = pd.Timestamp(row["sat_date"])
    start = ee.Date(sat_date.strftime("%Y-%m-%d"))
    end = start.advance(1, "day")
    collection = (
        ee.ImageCollection(S2_COLLECTION)
        .filterBounds(geometry)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_SCENE_CLOUD_PCT))
    )
    if collection.size().getInfo() == 0:
        raise ValueError(f"No S2 scenes for {row['map_id']} on {sat_date.date()}")

    prepared = collection.map(prepare_s2).mosaic()
    target_projection = prepared.select("B3").projection()
    water = (
        prepared.select("water_mask").unmask(0).gt(0)
        .reproject(crs=target_projection, scale=EXPORT_SCALE_M)
    )
    clear = (
        prepared.select("valid_mask").unmask(0).gt(0)
        .reproject(crs=target_projection, scale=EXPORT_SCALE_M)
    )
    base_water = (
        prepared.select(BASE_FEATURES).resample("bilinear")
        .reproject(crs=target_projection, scale=EXPORT_SCALE_M)
        .updateMask(water)
    )

    moving = base_water.reduceNeighborhood(
        reducer=ee.Reducer.median(), kernel=WINDOW_KERNEL, skipMasked=True
    ).rename([f"{name}_median" for name in BASE_FEATURES])

    one = ee.Image.constant(1).rename("one").reproject(
        crs=target_projection, scale=EXPORT_SCALE_M
    )
    total_count = one.reduceNeighborhood(ee.Reducer.sum(), WINDOW_KERNEL).rename("total_pixel_count")
    clear_count = clear.rename("clear").reduceNeighborhood(ee.Reducer.sum(), WINDOW_KERNEL).rename("valid_pixel_count")
    water_count = water.rename("water").reduceNeighborhood(ee.Reducer.sum(), WINDOW_KERNEL).rename("valid_water_pixel_count")
    water_fraction = water_count.divide(clear_count.max(1)).clamp(0, 1).rename("water_pixel_fraction")
    water_pct = water_fraction.multiply(100).rename("water_pixel_pct")
    local_cloud_pct = ee.Image(1).subtract(clear_count.divide(total_count)).clamp(0, 1).multiply(100).rename("local_cloud_pct")
    candidate_water_count = water_count.rename("candidate_water_count")
    scene_cloud_pct = ee.Image.constant(collection.aggregate_mean("CLOUDY_PIXEL_PERCENTAGE")).rename("scene_cloud_pct")

    # Custom SS features are computed from moving-window band medians, matching the pandas training code.
    b2, b3, b4 = moving.select("B2_median"), moving.select("B3_median"), moving.select("B4_median")
    b8, b11, b12 = moving.select("B8_median"), moving.select("B11_median"), moving.select("B12_median")
    custom = ee.Image.cat([
        normalized_difference(b3, b8, "NDWI1"),
        normalized_difference(b8, b11, "NDWI2"),
        normalized_difference(b8, b12, "NDWI3"),
        safe_ratio(b4, b3, "B4_B3_ratio"),
        safe_ratio(b8, b4, "B8_B4_ratio"),
        safe_ratio(b8, b2, "NIR_Blue_ratio"),
        safe_ratio(b8, b3, "NIR_Green_ratio"),
        safe_ratio(b8, b4, "NIR_Red_ratio"),
        normalized_difference(b4, b8, "Red_NIR_ND"),
    ])

    acquisition = ee.Date(collection.aggregate_min("system:time_start"))
    cutoff = ee.Date.parse("YYYY-MM-dd HH", acquisition.format("YYYY-MM-dd HH"))
    temporal_context = make_temporal_context(cutoff)

    qa = ee.Image.cat([
        water_count, clear_count, total_count, water_fraction, water_pct,
        candidate_water_count, local_cloud_pct, scene_cloud_pct,
    ])
    prediction_support = water.And(water_count.gte(MIN_WINDOW_WATER_PIXELS)).rename("prediction_support")

    stack = (
        moving.addBands(custom).addBands(qa)
        .addBands(STATIC_CONTEXT).addBands(temporal_context)
        .addBands(prediction_support)
        .updateMask(prediction_support)
        .clip(geometry)
        .toFloat()
    )
    metadata = {
        "n_source_scenes": collection.size().getInfo(),
        "source_scene_ids": collection.aggregate_array("system:index").getInfo(),
        "acquisition_cutoff_utc": cutoff.format("YYYY-MM-dd'T'HH:mm:ss").getInfo(),
    }
    return stack, prepared, metadata

## 6. Extract contextual conditions and rank date pairs

In [ ]:
def candidate_collection(sat_date):
    start = ee.Date(pd.Timestamp(sat_date).strftime("%Y-%m-%d"))
    return (
        ee.ImageCollection(S2_COLLECTION)
        .filterBounds(TARGET_GEOMETRY)
        .filterDate(start, start.advance(1, "day"))
        .filter(
            ee.Filter.lt(
                "CLOUDY_PIXEL_PERCENTAGE", MAX_SCENE_CLOUD_PCT
            )
        )
    )


context_audit_rows = []
for row in candidates.to_dict("records"):
    collection = candidate_collection(row["sat_date"])
    n_scenes = collection.size().getInfo()
    if n_scenes == 0:
        print("Skipping date without usable S2 scene:", row["sat_date"])
        continue
    acquisition = ee.Date(
        collection.aggregate_min("system:time_start")
    )
    cutoff = ee.Date.parse(
        "YYYY-MM-dd HH", acquisition.format("YYYY-MM-dd HH")
    )
    temporal = make_temporal_context(cutoff).select(
        DYNAMIC_CONTEXT_FEATURES
    )
    values = temporal.reduceRegion(
        reducer=ee.Reducer.median(),
        geometry=TARGET_GEOMETRY,
        scale=COARSE_CONTEXT_SCALE_M,
        maxPixels=1e7,
        bestEffort=True,
    ).getInfo()
    context_audit_rows.append({
        "map_id": TARGET_MAP_ID,
        "coordinate_waterbody": TARGET_WATERBODY,
        "sat_date": pd.Timestamp(row["sat_date"]).date().isoformat(),
        "acquisition_cutoff_utc": cutoff.format(
            "YYYY-MM-dd'T'HH:mm:ss"
        ).getInfo(),
        "n_source_scenes": n_scenes,
        "scene_cloud_pct_mean": collection.aggregate_mean(
            "CLOUDY_PIXEL_PERCENTAGE"
        ).getInfo(),
        "clear_static_water_coverage_pct": row[
            "clear_static_water_coverage_pct"
        ],
        "clear_static_water_pixels": row[
            "clear_static_water_pixels"
        ],
        **values,
    })
    print("Context extracted:", row["sat_date"])

context_audit = pd.DataFrame(context_audit_rows)
if len(context_audit) < 2:
    raise ValueError("Fewer than two candidate dates have complete context.")

# Robust scaling comes from the existing context dataset, not from
# the five candidate dates and not from model predictions.
robust_scale = {}
for feature in DYNAMIC_CONTEXT_FEATURES:
    values = context[feature].dropna().astype(float)
    median = float(values.median())
    q25, q75 = values.quantile([0.25, 0.75])
    scale = float(q75 - q25)
    if not np.isfinite(scale) or scale <= 0:
        scale = float(values.std(ddof=0))
    if not np.isfinite(scale) or scale <= 0:
        scale = 1.0
    robust_scale[feature] = {"median": median, "scale": scale}
    context_audit[f"z_{feature}"] = (
        pd.to_numeric(context_audit[feature], errors="coerce") - median
    ) / scale

pair_rows = []
for left_index, right_index in combinations(context_audit.index, 2):
    left = context_audit.loc[left_index]
    right = context_audit.loc[right_index]
    date_left = pd.Timestamp(left["sat_date"])
    date_right = pd.Timestamp(right["sat_date"])
    separation_days = abs((date_right - date_left).days)
    if separation_days < MIN_DATE_SEPARATION_DAYS:
        continue

    group_distances = {}
    for group, features in CONTEXT_GROUPS.items():
        differences = [
            abs(left[f"z_{feature}"] - right[f"z_{feature}"])
            for feature in features
            if np.isfinite(left[f"z_{feature}"])
            and np.isfinite(right[f"z_{feature}"])
        ]
        group_distances[group] = (
            float(np.mean(differences)) if differences else np.nan
        )
    valid_group_distances = [
        value for value in group_distances.values()
        if np.isfinite(value)
    ]
    contrast = float(np.mean(valid_group_distances))
    coverage_weight = np.sqrt(
        max(left["clear_static_water_coverage_pct"], 0)
        * max(right["clear_static_water_coverage_pct"], 0)
    ) / 100.0
    pair_rows.append({
        "date_1": date_left.date().isoformat(),
        "date_2": date_right.date().isoformat(),
        "separation_days": separation_days,
        "rainfall_distance": group_distances["rainfall"],
        "runoff_distance": group_distances["runoff"],
        "meteorology_distance": group_distances["meteorology"],
        "context_contrast_score": contrast,
        "minimum_clear_water_coverage_pct": min(
            left["clear_static_water_coverage_pct"],
            right["clear_static_water_coverage_pct"],
        ),
        "coverage_weight": coverage_weight,
        "selection_score": contrast * coverage_weight,
        "selection_used_predictions": False,
    })

pair_ranking = pd.DataFrame(pair_rows).sort_values(
    ["selection_score", "context_contrast_score"],
    ascending=[False, False],
).reset_index(drop=True)
if pair_ranking.empty:
    raise ValueError("No candidate pair passed the date-separation rule.")
pair_ranking.insert(0, "pair_rank", np.arange(1, len(pair_ranking) + 1))

context_audit.to_csv(
    OUTPUT_DIR / "figure2_candidate_date_context_audit.csv",
    index=False, encoding="utf-8-sig",
)
pair_ranking.to_csv(
    OUTPUT_DIR / "figure2_candidate_pair_ranking.csv",
    index=False, encoding="utf-8-sig",
)
display(context_audit[[
    "sat_date", "acquisition_cutoff_utc", "n_source_scenes",
    "scene_cloud_pct_mean", "clear_static_water_coverage_pct",
    *DYNAMIC_CONTEXT_FEATURES,
]].round(3))
display(pair_ranking.head(10).round(4))

## 7. Lock the reviewed pair and build the two-date manifest

In [ ]:
if DATE_PAIR_OVERRIDE is None:
    selected_dates = [
        pair_ranking.iloc[0]["date_1"],
        pair_ranking.iloc[0]["date_2"],
    ]
    selection_source = "highest preregistered context-and-coverage score"
else:
    if len(DATE_PAIR_OVERRIDE) != 2:
        raise ValueError("DATE_PAIR_OVERRIDE must contain exactly two dates.")
    selected_dates = sorted(
        pd.Timestamp(value).date().isoformat()
        for value in DATE_PAIR_OVERRIDE
    )
    selection_source = "manual override after candidate audit"

available_dates = set(context_audit["sat_date"])
missing_dates = sorted(set(selected_dates) - available_dates)
if missing_dates:
    raise ValueError(
        f"Selected dates are not available in the audit: {missing_dates}"
    )

selected_pair_rows = context_audit.loc[
    context_audit["sat_date"].isin(selected_dates)
].copy()
if len(selected_pair_rows) != 2:
    raise ValueError("The selected pair did not resolve to exactly two dates.")

export_manifest = selected_pair_rows[[
    "map_id", "coordinate_waterbody", "sat_date",
    "acquisition_cutoff_utc", "n_source_scenes",
    "scene_cloud_pct_mean", "clear_static_water_coverage_pct",
    "clear_static_water_pixels", *DYNAMIC_CONTEXT_FEATURES,
]].copy()
export_manifest["job_id"] = (
    export_manifest["map_id"] + "_"
    + export_manifest["sat_date"].str.replace("-", "", regex=False)
)
export_manifest["selection_source"] = selection_source
export_manifest["selection_used_predictions"] = False
export_manifest = export_manifest.sort_values("sat_date").reset_index(drop=True)
export_manifest.to_csv(
    OUTPUT_DIR / "figure2_selected_date_pair_manifest.csv",
    index=False, encoding="utf-8-sig",
)

print("Selected dates:", selected_dates)
print("Selection source:", selection_source)
display(export_manifest.round(4))
if not START_EXPORT_TASKS:
    print(
        "Review the pair above. To export it, set "
        "START_EXPORT_TASKS=True in the configuration cell, rerun that "
        "cell, and then rerun Sections 7-11."
    )

## 8. Build both 62-band feature stacks on the reviewed AOI

In [ ]:
jobs = {}
job_rows = []
expected_schema = None

for row in export_manifest.to_dict("records"):
    stack, prepared, metadata = make_feature_stack(row, TARGET_GEOMETRY)
    schema = stack.bandNames().getInfo()
    if expected_schema is None:
        expected_schema = schema
    elif schema != expected_schema:
        raise ValueError(f"Band schema mismatch for {row['job_id']}")

    jobs[row["job_id"]] = {
        "row": row,
        "geometry": TARGET_GEOMETRY,
        "stack": stack,
        "prepared": prepared,
        "metadata": metadata,
    }
    job_rows.append({
        "job_id": row["job_id"],
        "map_id": row["map_id"],
        "coordinate_waterbody": row["coordinate_waterbody"],
        "sat_date": row["sat_date"],
        "aoi_source": "reviewed_drawn_polygon",
        "aoi_area_km2": TARGET_GEOMETRY.area(maxError=1).divide(1e6).getInfo(),
        "n_bands": len(schema),
        **metadata,
    })
    print(row["job_id"], "bands:", len(schema))

if len(expected_schema) != 62 or len(set(expected_schema)) != 62:
    raise ValueError(
        f"Expected 62 unique bands, found {len(expected_schema)}"
    )
job_audit = pd.DataFrame(job_rows)
band_schema = pd.DataFrame({
    "band_order": range(1, len(expected_schema) + 1),
    "band": expected_schema,
})
job_audit.to_csv(
    OUTPUT_DIR / "figure2_moving_window_job_audit.csv",
    index=False, encoding="utf-8-sig",
)
band_schema.to_csv(
    OUTPUT_DIR / "figure2_moving_window_band_schema.csv",
    index=False, encoding="utf-8-sig",
)
display(job_audit)
display(band_schema)

## 9. Lock the Earth Engine export grid to the existing WB02 raster

In [ ]:
with rasterio.open(REFERENCE_GRID_PATH) as reference:
    reference_crs = reference.crs
    reference_transform = reference.transform
    reference_width = reference.width
    reference_height = reference.height
    reference_band_count = reference.count

if reference_crs is None:
    raise ValueError("Reference grid has no CRS.")
if abs(reference_transform.b) > 1e-12 or abs(reference_transform.d) > 1e-12:
    raise ValueError("Rotated reference transforms are not supported.")

EXPORT_CRS = reference_crs.to_string()
EXPORT_CRS_TRANSFORM = [
    reference_transform.a,
    reference_transform.b,
    reference_transform.c,
    reference_transform.d,
    reference_transform.e,
    reference_transform.f,
]
reference_grid = pd.DataFrame([{
    "reference_file": str(REFERENCE_GRID_PATH),
    "crs": EXPORT_CRS,
    "transform": tuple(EXPORT_CRS_TRANSFORM),
    "width": reference_width,
    "height": reference_height,
    "band_count": reference_band_count,
}])
reference_grid.to_csv(
    OUTPUT_DIR / "figure2_reference_grid.csv",
    index=False, encoding="utf-8-sig",
)
display(reference_grid)

## 10. Export two dates: feature stack and Sentinel-2 RGB

In [ ]:
export_tasks = {}

if START_EXPORT_TASKS:
    for job_id, job in jobs.items():
        row = job["row"]
        date_text = pd.Timestamp(row["sat_date"]).strftime("%Y%m%d")
        map_id = row["map_id"]

        feature_prefix = f"{map_id}_{date_text}_moving_window_features"
        feature_task = ee.batch.Export.image.toDrive(
            image=job["stack"],
            description=feature_prefix,
            folder=DRIVE_EXPORT_FOLDER,
            fileNamePrefix=feature_prefix,
            region=job["geometry"],
            crs=EXPORT_CRS,
            crsTransform=EXPORT_CRS_TRANSFORM,
            maxPixels=1e10,
            fileFormat="GeoTIFF",
            formatOptions={
                "cloudOptimized": True,
                "noData": -9999,
            },
        )
        feature_task.start()
        export_tasks[(job_id, "features")] = feature_task

        rgb_prefix = f"{map_id}_{date_text}_sentinel2_rgb"
        clear_rgb = (
            job["prepared"].select(["B4", "B3", "B2"])
            .updateMask(job["prepared"].select("valid_mask"))
            .clip(job["geometry"])
            .toFloat()
        )
        rgb_task = ee.batch.Export.image.toDrive(
            image=clear_rgb,
            description=rgb_prefix,
            folder=RGB_DRIVE_EXPORT_FOLDER,
            fileNamePrefix=rgb_prefix,
            region=job["geometry"],
            crs=EXPORT_CRS,
            crsTransform=EXPORT_CRS_TRANSFORM,
            maxPixels=1e10,
            fileFormat="GeoTIFF",
            formatOptions={
                "cloudOptimized": True,
                "noData": -9999,
            },
        )
        rgb_task.start()
        export_tasks[(job_id, "sentinel2_rgb")] = rgb_task
        print(
            "Started:", job_id,
            "features=", feature_task.id,
            "rgb=", rgb_task.id,
        )
else:
    print(
        "START_EXPORT_TASKS=False. Pair screening and stack audits are "
        "complete, but no export was started."
    )

In [ ]:
def task_status_table(tasks):
    rows = []
    for (job_id, task_type), task in tasks.items():
        status = task.status()
        rows.append({
            "job_id": job_id,
            "task_type": task_type,
            "task_id": task.id,
            "state": status.get("state"),
            "error_message": status.get("error_message"),
        })
    return pd.DataFrame(rows).sort_values(
        ["job_id", "task_type"]
    ).reset_index(drop=True)


if export_tasks:
    display(task_status_table(export_tasks))

if WAIT_FOR_EXPORTS and export_tasks:
    deadline = time.time() + MAX_WAIT_MINUTES * 60
    terminal = {"COMPLETED", "FAILED", "CANCELLED"}
    while time.time() < deadline:
        status_df = task_status_table(export_tasks)
        display(status_df)
        if status_df["state"].isin(terminal).all():
            break
        time.sleep(POLL_SECONDS)

if export_tasks:
    final_export_status = task_status_table(export_tasks)
    final_export_status.to_csv(
        OUTPUT_DIR / "figure2_export_task_status.csv",
        index=False, encoding="utf-8-sig",
    )
    display(final_export_status)

## 11. Verify the hand-off for the Figure 2 prediction notebook

In [ ]:
FEATURE_EXPORT_DIR = MYDRIVE / DRIVE_EXPORT_FOLDER
RGB_EXPORT_DIR = MYDRIVE / RGB_DRIVE_EXPORT_FOLDER
expected_jobs = set(export_manifest["job_id"])


def raster_inventory(root, token, expected_count):
    rows = []
    for path in sorted(root.rglob("*.tif")) if root.exists() else []:
        if token not in path.name:
            continue
        match = re.match(
            rf"^({TARGET_MAP_ID})_(\d{{8}})_", path.name
        )
        if match is None:
            continue
        job_id = f"{match.group(1)}_{match.group(2)}"
        if job_id not in expected_jobs:
            continue
        try:
            with rasterio.open(path) as source:
                transform = tuple(source.transform)[:6]
                rows.append({
                    "job_id": job_id,
                    "file": str(path),
                    "band_count": source.count,
                    "width": source.width,
                    "height": source.height,
                    "crs": str(source.crs),
                    "transform": transform,
                    "schema_valid": source.count == expected_count,
                    "grid_matches_reference": (
                        str(source.crs) == EXPORT_CRS
                        and np.allclose(
                            transform,
                            tuple(EXPORT_CRS_TRANSFORM),
                            rtol=0,
                            atol=1e-12,
                        )
                        and source.width == reference_width
                        and source.height == reference_height
                    ),
                    "error": None,
                })
        except Exception as error:
            rows.append({
                "job_id": job_id, "file": str(path),
                "band_count": np.nan, "width": np.nan,
                "height": np.nan, "crs": None,
                "transform": None, "schema_valid": False,
                "grid_matches_reference": False,
                "error": str(error),
            })
    return pd.DataFrame(rows)


feature_inventory = raster_inventory(
    FEATURE_EXPORT_DIR, "moving_window_features", len(expected_schema)
)
rgb_inventory = raster_inventory(
    RGB_EXPORT_DIR, "sentinel2_rgb", 3
)
feature_inventory.to_csv(
    OUTPUT_DIR / "figure2_feature_geotiff_inventory.csv",
    index=False, encoding="utf-8-sig",
)
rgb_inventory.to_csv(
    OUTPUT_DIR / "figure2_rgb_geotiff_inventory.csv",
    index=False, encoding="utf-8-sig",
)

if export_tasks:
    current_status = task_status_table(export_tasks)
    tasks_complete = current_status["state"].eq("COMPLETED").all()
    task_detail = current_status["state"].value_counts().to_dict()
else:
    tasks_complete = None
    task_detail = "No in-memory tasks; verify completion from Drive files."

feature_valid_jobs = set(
    feature_inventory.loc[
        feature_inventory.get("schema_valid", False).eq(True)
        & feature_inventory.get("grid_matches_reference", False).eq(True),
        "job_id",
    ]
) if not feature_inventory.empty else set()
rgb_valid_jobs = set(
    rgb_inventory.loc[
        rgb_inventory.get("schema_valid", False).eq(True)
        & rgb_inventory.get("grid_matches_reference", False).eq(True),
        "job_id",
    ]
) if not rgb_inventory.empty else set()

if tasks_complete is None:
    tasks_complete = (
        expected_jobs.issubset(feature_valid_jobs)
        and expected_jobs.issubset(rgb_valid_jobs)
    )
    task_detail = (
        "No in-memory tasks; completion verified from exported GeoTIFFs."
        if tasks_complete
        else "No in-memory tasks and one or more exported files are missing."
    )

checks = [
    {
        "check": "two_dates_selected",
        "passed": len(export_manifest) == 2,
        "detail": ", ".join(export_manifest["sat_date"]),
    },
    {
        "check": "selection_independent_of_predictions",
        "passed": not export_manifest["selection_used_predictions"].any(),
        "detail": export_manifest["selection_source"].iloc[0],
    },
    {
        "check": "reviewed_polygon_loaded",
        "passed": polygon_feature is not None,
        "detail": TARGET_MAP_ID,
    },
    {
        "check": "same_62_band_schema",
        "passed": len(expected_schema) == 62 and len(set(expected_schema)) == 62,
        "detail": f"{len(expected_schema)} bands",
    },
    {
        "check": "earth_engine_exports_completed",
        "passed": bool(tasks_complete),
        "detail": str(task_detail),
    },
    {
        "check": "two_feature_geotiffs_valid_on_reference_grid",
        "passed": expected_jobs.issubset(feature_valid_jobs),
        "detail": f"{len(expected_jobs & feature_valid_jobs)}/2 jobs",
    },
    {
        "check": "two_rgb_geotiffs_valid_on_reference_grid",
        "passed": expected_jobs.issubset(rgb_valid_jobs),
        "detail": f"{len(expected_jobs & rgb_valid_jobs)}/2 jobs",
    },
]
readiness = pd.DataFrame(checks)
readiness.to_csv(
    OUTPUT_DIR / "figure2_preparation_readiness.csv",
    index=False, encoding="utf-8-sig",
)
display(feature_inventory)
display(rgb_inventory)
display(readiness)
print("FIGURE_2_DATA_READY =", bool(readiness["passed"].all()))

## Handoff to the Figure 2 prediction notebook

Continue only when `FIGURE_2_DATA_READY = True`. Keep these files:

- `figure2_candidate_date_context_audit.csv`
- `figure2_candidate_pair_ranking.csv`
- `figure2_selected_date_pair_manifest.csv`
- both 62-band feature GeoTIFFs
- both Sentinel-2 RGB GeoTIFFs
- `figure2_reference_grid.csv`
- `figure2_preparation_readiness.csv`